In [2]:
# Imports
import tensorflow as tf
import pandas as pd
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential

In [3]:
# Import train,val, and test data from preprocessing.ipynp

train_df = pd.read_csv("../working/train.csv")
val_df = pd.read_csv("../working/val.csv")
test_df = pd.read_csv("../working/test.csv")

In [4]:
# data augmentation layers in the model
data_augmentation = Sequential([
    layers.RandomBrightness(factor=0.2,  value_range=(0, 255)),
    layers.RandomFlip(mode='horizontal_and_vertical'),
    layers.RandomRotation(factor=0.2)
], name='data_augmentation')

In [5]:
# turn train, val, and test dataframes to images for model input

def make_ds(df, shuffle=False, batch=32):
    ds = tf.data.Dataset.from_tensor_slices((df['path'].values, df['diagnosis'].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    def load(path, label):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        return tf.cast(img, tf.float32), label   # keep 0–255 values
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

In [6]:
# Pretrained VGG16 feature extractor
base = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained layers
base.trainable = False

In [7]:
# Build the complete model
model = Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Data augmentation
    data_augmentation,

    # VGG16 preprocessing
    layers.Lambda(
        tf.keras.applications.vgg16.preprocess_input
    ),

    # Feature extraction
    base,

    # Classification head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy",
         tf.keras.metrics.AUC(name="auc"),
         tf.keras.metrics.Precision(name="precision"),
         tf.keras.metrics.Recall(name="recall")]
)

# callback functions
cbs = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("../output/vgg16_f3_head.weights.h5", monitor="val_auc",
                                       mode="max", save_best_only=True, save_weights_only=True),
]



model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [8]:
print(tf.config.list_physical_devices('GPU'))

[]


In [9]:
# train fully connected layers with last convolution block frozen

history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Epoch 1/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 680s 3s/step - accuracy: 0.6620 - auc: 0.7084 - loss: 0.8137 - precision: 0.6647 - recall: 0.6536 - val_accuracy: 0.7718 - val_auc: 0.8651 - val_loss: 0.5238 - val_precision: 0.9005 - val_recall: 0.7235
Epoch 2/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 1660s 8s/step - accuracy: 0.7644 - auc: 0.8300 - loss: 0.5138 - precision: 0.7571 - recall: 0.7787 - val_accuracy: 0.8008 - val_auc: 0.8944 - val_loss: 0.4632 - val_precision: 0.9121 - val_recall: 0.7624
Epoch 3/30
104/198 ━━━━━━━━━━━━━━━━━━━━ 5:26 3s/step - accuracy: 0.7826 - auc: 0.8632 - loss: 0.4615 - precision: 0.7927 - recall: 0.7774

KeyboardInterrupt: 

In [ ]:
# fine tune last convolution block

base.trainable = True
for l in base.layers:
    l.trainable = l.name.startswith("block5")      # unfreeze only the last conv block

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),   # much smaller learning rate
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])

cbs[1] = tf.keras.callbacks.ModelCheckpoint("../output/vgg16_f3_ft.weights.h5", monitor="val_auc",
                                            mode="max", save_best_only=True, save_weights_only=True)
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)